# Restartable RLMF Colab Orchestrator

This notebook runs the checked-in RLMF package without implementing training or evaluation logic. It checks out one exact project commit, verifies the pinned runtime and manifests, uses local Colab scratch space for all working artifacts, and exports manifest-bound checkpoint archives after completed stages.


## Select one run mode

- `smoke`: the registered smoke config and seed 11. Runtime outputs are infrastructure evidence only.
- `pilot`: the registered confirmatory config, stopped at step 25 by the Task 9 command. Runtime outputs are infrastructure evidence only.
- `confirmatory`: the unchanged confirmatory config and one selected registered seed.

Set `RUN_MODE`, `PROJECT_COMMIT`, and optionally `PROJECT_SOURCE`, `SELECTED_SEED`, `USE_DRIVE=1`, and `RUN_EVALUATION=1` in the Colab environment before running from the top. `PROJECT_SOURCE` must be a cloneable repository URL or an uploaded Git repository.


In [ ]:
from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

RUN_MODE = os.environ.get("RUN_MODE", "smoke")
USE_DRIVE = os.environ.get("USE_DRIVE", "0") == "1"
PROJECT_COMMIT = os.environ.get("PROJECT_COMMIT", "")
PROJECT_SOURCE = os.environ.get("PROJECT_SOURCE", "/content/project-upload")
PROJECT_ROOT = Path("/content/metacognitive-feature-flow")
MODE_CONFIGS = {
    "smoke": "configs/rlmf_qwen06b_smoke.json",
    "pilot": "configs/rlmf_qwen06b_confirmatory.json",
    "confirmatory": "configs/rlmf_qwen06b_confirmatory.json",
}
if RUN_MODE not in MODE_CONFIGS:
    raise ValueError(f"RUN_MODE must be one of {sorted(MODE_CONFIGS)}")
CONFIG_PATH = MODE_CONFIGS[RUN_MODE]
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise ValueError("PROJECT_COMMIT must be an exact lowercase 40-character Git SHA")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--no-checkout", PROJECT_SOURCE, str(PROJECT_ROOT)], check=True)
if not (PROJECT_ROOT / ".git").exists():
    raise RuntimeError("PROJECT_ROOT must be a Git repository")
subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", "--detach", PROJECT_COMMIT], check=True)
resolved_commit = subprocess.check_output(["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True).strip()
if resolved_commit != PROJECT_COMMIT:
    raise RuntimeError("checked-out commit does not match PROJECT_COMMIT")
dirty_paths = subprocess.check_output(["git", "-C", str(PROJECT_ROOT), "status", "--porcelain"], text=True).strip()
if dirty_paths:
    raise RuntimeError(f"project checkout is dirty: {dirty_paths}")
os.chdir(PROJECT_ROOT)
print(json.dumps({"project_commit": resolved_commit, "run_mode": RUN_MODE}, sort_keys=True))


## Install and verify the frozen runtime

Dependencies come only from `requirements-rlmf-colab.txt`. The project is then installed editable without dependency resolution, and the package performs the runtime version check.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--requirement", "requirements-rlmf-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)

from trajectory_extractor.rlmf_artifacts import RLMFArtifactStore
from trajectory_extractor.rlmf_training import import_checkpoint, validate_runtime_versions
from trajectory_extractor.rlmf_types import CheckpointRecord, RLMFConfig

runtime_versions = validate_runtime_versions()
print(json.dumps({"runtime_versions": runtime_versions}, sort_keys=True))


## Configure isolated storage

Training and checkpoint construction stay under `/content/rlmf-scratch`. Drive is mounted only for archive import/export when `USE_DRIVE=1`; no live artifact root points at Drive.


In [ ]:
config = RLMFConfig.from_json(CONFIG_PATH)
SELECTED_SEED = int(os.environ.get("SELECTED_SEED", "11"))
if SELECTED_SEED not in config.seeds:
    raise ValueError("SELECTED_SEED is not registered in the selected config")
expected_profile = "smoke" if RUN_MODE == "smoke" else "confirmatory"
if config.profile != expected_profile:
    raise RuntimeError("run mode and checked-in config profile do not match")
SCRATCH_ROOT = Path("/content/rlmf-scratch")
ARTIFACT_ROOT = SCRATCH_ROOT / "artifacts"
LOCAL_EXPORT_ROOT = SCRATCH_ROOT / "exports" / PROJECT_COMMIT / RUN_MODE
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_EXPORT_ROOT = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_EXPORT_ROOT = Path("/content/drive/MyDrive/rlmf-checkpoints") / PROJECT_COMMIT / RUN_MODE
    DRIVE_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
store = RLMFArtifactStore(ARTIFACT_ROOT)
print(json.dumps({"artifact_root": str(ARTIFACT_ROOT), "config": CONFIG_PATH, "drive_enabled": USE_DRIVE, "seed": SELECTED_SEED}, sort_keys=True))


## Enforce the Colab GPU gate

The run stops before data or model work unless CUDA is available and the selected GPU reports at least 14 GB of total VRAM.


In [ ]:
import torch

MIN_GPU_MEMORY_GB = 14
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required")
gpu_properties = torch.cuda.get_device_properties(0)
gpu_memory_gb = gpu_properties.total_memory / (1024 ** 3)
if gpu_memory_gb < MIN_GPU_MEMORY_GB:
    raise RuntimeError(f"GPU has {gpu_memory_gb:.2f} GB; at least {MIN_GPU_MEMORY_GB} GB is required")
print(json.dumps({"gpu_name": gpu_properties.name, "gpu_memory_gb": round(gpu_memory_gb, 3), "minimum_gpu_memory_gb": MIN_GPU_MEMORY_GB}, sort_keys=True))


## Verify upstream manifests and sealed data

The checked-in preregistration tests verify vendored upstream hashes. Data preparation is delegated to the CLI only when the completion marker is absent; every path then goes through `RLMFArtifactStore.verify_endpoint`.


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "tests/test_rlmf_preregistration.py", "-q"], check=True)
prepare_marker = ARTIFACT_ROOT / "runs" / "rlmf" / config.study_id / "endpoints" / "prepare-data.complete.json"
if not prepare_marker.exists():
    subprocess.run(["feature-dynamics", "rlmf-prepare-data", "--config", CONFIG_PATH, "--root", str(ARTIFACT_ROOT)], check=True)
data_manifest = store.verify_endpoint(config.study_id, "prepare-data")
print(json.dumps({"data_endpoint": data_manifest["endpoint"], "data_parent_hashes": data_manifest["parent_hashes"]}, sort_keys=True))


## Run and export checkpoints in the registered order

The package owns all training behavior. This cell runs pre-SFT once, then `standard`, then `rlmf`. Existing completed stages are verified and skipped; incomplete verified stages receive `--resume`. Each completed checkpoint is exported by the CLI to local scratch before atomic publication. Existing archives are verified against the checkpoint hash and never overwritten.


In [ ]:
PILOT_STEPS = 25
TRAINING_STAGES = [
    {"name": "pre_sft", "record_stage": "pre_sft", "record_arm": None, "seed": None, "cli": ["feature-dynamics", "rlmf-train", "--config", CONFIG_PATH, "--artifact-root", str(ARTIFACT_ROOT), "--stage", "pre-sft"]},
    {"name": "standard", "record_stage": "rl", "record_arm": "standard_grpo", "seed": SELECTED_SEED, "cli": ["feature-dynamics", "rlmf-train", "--config", CONFIG_PATH, "--artifact-root", str(ARTIFACT_ROOT), "--stage", "rl", "--arm", "standard", "--seed", str(SELECTED_SEED)]},
    {"name": "rlmf", "record_stage": "rl", "record_arm": "rlmf", "seed": SELECTED_SEED, "cli": ["feature-dynamics", "rlmf-train", "--config", CONFIG_PATH, "--artifact-root", str(ARTIFACT_ROOT), "--stage", "rl", "--arm", "rlmf", "--seed", str(SELECTED_SEED)]},
]
if RUN_MODE == "pilot":
    for stage in TRAINING_STAGES[1:]:
        stage["cli"].extend(["--stop-after-step", str(PILOT_STEPS)])
checkpoint_summary = {}
checkpoint_root = ARTIFACT_ROOT / "runs" / "rlmf" / config.study_id / "checkpoints"
for stage in TRAINING_STAGES:
    archive_name = f"{config.study_id}-{stage['name']}-{stage['seed'] if stage['seed'] is not None else 'shared'}.tar"
    local_target = LOCAL_EXPORT_ROOT / archive_name
    drive_target = DRIVE_EXPORT_ROOT / archive_name if USE_DRIVE else None
    records = [CheckpointRecord.from_record(json.loads(path.read_text())) for path in sorted(checkpoint_root.glob("*/checkpoint.json"))]
    matching = [record for record in records if record.stage == stage["record_stage"] and record.arm == stage["record_arm"] and record.seed == stage["seed"]]
    if not any(record.completed for record in matching) and USE_DRIVE and drive_target.exists():
        subprocess.run(["feature-dynamics", "rlmf-import-checkpoint", "--artifact-root", str(ARTIFACT_ROOT), "--archive", str(drive_target)], check=True)
        records = [CheckpointRecord.from_record(json.loads(path.read_text())) for path in sorted(checkpoint_root.glob("*/checkpoint.json"))]
        matching = [record for record in records if record.stage == stage["record_stage"] and record.arm == stage["record_arm"] and record.seed == stage["seed"]]
    completed = [record for record in matching if record.completed]
    if completed:
        record = max(completed, key=lambda item: (item.global_step, item.micro_step))
        stage_status = "verified_existing"
    else:
        command = list(stage["cli"])
        if matching:
            command.append("--resume")
        process = subprocess.run(command, check=True, capture_output=True, text=True)
        print(process.stdout, end="")
        command_result = json.loads(process.stdout.strip().splitlines()[-1])
        record = CheckpointRecord.from_record(json.loads((Path(command_result["checkpoint"]) / "checkpoint.json").read_text()))
        stage_status = "resumed" if matching else "completed"
    if not record.completed:
        raise RuntimeError(f"stage did not produce a completed checkpoint: {stage['name']}")
    if local_target.exists():
        with tempfile.TemporaryDirectory(dir=SCRATCH_ROOT) as verify_root:
            verified_archive = import_checkpoint(RLMFArtifactStore(Path(verify_root)), local_target)
        if verified_archive.checkpoint_hash != record.checkpoint_hash:
            raise FileExistsError(f"existing local archive belongs to another checkpoint: {local_target}")
    else:
        with tempfile.TemporaryDirectory(dir=SCRATCH_ROOT) as export_root:
            local_archive = Path(export_root) / archive_name
            subprocess.run(["feature-dynamics", "rlmf-export-checkpoint", "--artifact-root", str(ARTIFACT_ROOT), "--checkpoint", str(record.path), "--output", str(local_archive)], check=True)
            os.replace(local_archive, local_target)
    if USE_DRIVE:
        if drive_target.exists():
            with tempfile.TemporaryDirectory(dir=SCRATCH_ROOT) as verify_root:
                verified_archive = import_checkpoint(RLMFArtifactStore(Path(verify_root)), drive_target)
            if verified_archive.checkpoint_hash != record.checkpoint_hash:
                raise FileExistsError(f"existing Drive archive belongs to another checkpoint: {drive_target}")
        else:
            with tempfile.NamedTemporaryFile(dir=DRIVE_EXPORT_ROOT, prefix=f".{archive_name}.", suffix=".tmp", delete=False) as partial:
                partial_path = Path(partial.name)
            shutil.copyfile(local_target, partial_path)
            with partial_path.open("rb") as handle:
                os.fsync(handle.fileno())
            if drive_target.exists():
                partial_path.unlink()
                raise FileExistsError(drive_target)
            os.replace(partial_path, drive_target)
    checkpoint_summary[stage["name"]] = {"archive": str(drive_target if USE_DRIVE else local_target), "checkpoint_hash": record.checkpoint_hash, "global_step": record.global_step, "status": stage_status}
print(json.dumps({"checkpoints": checkpoint_summary}, sort_keys=True))


## Optional registered evaluation handoff

Task 10 owns rollout generation and audit enforcement. When that CLI is present, set `RUN_EVALUATION=1` to delegate validation first and test second, with `standard` before `rlmf` in each split. The CLI must enforce the locked-audit prerequisite and produce the registered designated-plus-20 bundles for pilot/confirmatory configs. The smoke config retains its checked-in two-auxiliary scale.


In [ ]:
RUN_EVALUATION = os.environ.get("RUN_EVALUATION", "0") == "1"
EVALUATION_BUNDLE_KIND = "designated-plus-20" if RUN_MODE != "smoke" else "designated-plus-2"
EVALUATION_COMMANDS = [
    ["feature-dynamics", "rlmf-generate-rollouts", "--config", str(PROJECT_ROOT / CONFIG_PATH), "--arm", "standard", "--seed", str(SELECTED_SEED), "--split", "validation"],
    ["feature-dynamics", "rlmf-generate-rollouts", "--config", str(PROJECT_ROOT / CONFIG_PATH), "--arm", "rlmf", "--seed", str(SELECTED_SEED), "--split", "validation"],
    ["feature-dynamics", "rlmf-generate-rollouts", "--config", str(PROJECT_ROOT / CONFIG_PATH), "--arm", "standard", "--seed", str(SELECTED_SEED), "--split", "test"],
    ["feature-dynamics", "rlmf-generate-rollouts", "--config", str(PROJECT_ROOT / CONFIG_PATH), "--arm", "rlmf", "--seed", str(SELECTED_SEED), "--split", "test"],
]
if len(checkpoint_summary) != len(TRAINING_STAGES):
    raise RuntimeError("evaluation handoff requires every selected-seed checkpoint")
evaluation_status = "not_requested"
if RUN_EVALUATION:
    for command in EVALUATION_COMMANDS:
        subprocess.run(command, cwd=ARTIFACT_ROOT, check=True)
    evaluation_status = "commands_completed"
print(json.dumps({"evaluation_bundle_kind": EVALUATION_BUNDLE_KIND, "evaluation_status": evaluation_status}, sort_keys=True))


In [ ]:
completion_summary = {
    "artifact_root": str(ARTIFACT_ROOT),
    "checkpoint_hashes": {name: value["checkpoint_hash"] for name, value in checkpoint_summary.items()},
    "config": CONFIG_PATH,
    "drive_enabled": USE_DRIVE,
    "evaluation_status": evaluation_status,
    "project_commit": resolved_commit,
    "run_mode": RUN_MODE,
    "seed": SELECTED_SEED,
    "status": "orchestration_complete",
}
print(json.dumps(completion_summary, sort_keys=True))
